# 第四轮讨论：聚焦手型泛化

## 方向调整

用户决定暂时放弃物体泛化，只专注手型（embodiment）泛化。核心判断：
> "仅能实现手型的手内操作任务真正泛化，便已能超越 GET-Zero 并产出一篇成果了。物体泛化放到后续工作。"

这意味着前三轮中以下设计**暂时搁置**：
- object memory slots（1/2/4 个 slots 的讨论）
- hand↔object cross-attention
- per-finger local latent 对物体属性的估计
- local loss 预测接触/滑移、global loss 预测物体 extrinsics 的监督拆分

**保留**的核心共识：
- joint-level tokenization
- Graph Transformer 作为主干（graph-biased self-attention）
- dynamic state × static embodiment 的 cross-attention
- per-joint action head（joint-space 输出）
- history encoder 放图外

**新增聚焦点**：超越 GET-Zero 的具体切入，实验设计

## 4.1 补读论文和代码后：GET-Zero 的精确短板与我们的切入空间

**用户反馈**：要求先完整阅读 GET-Zero（正文 1-6 页）和 T(R,O) Grasp（正文 1-8 页 + 附录 11-12 页）论文及双方代码，再做分析。

**分析**：

### 一、GET-Zero 的实际 embodiment 编码——比想象中更弱

从代码 `embodiment_transformer.py` 和论文 Sec. III-A 可以精确还原 GET-Zero 每个 joint token 的构成：

$$
T_j = [\underbrace{o_{vl,j}}_{\text{joint angle/vel + PD target}},\; \underbrace{o_{vg}}_{\text{sinusoidal phase}},\; \underbrace{o_{fl,j}}_{\text{3D pos + rot from URDF}},\; \underbrace{o_{fg}}_{\text{(unused)}}]
$$

然后经过 `nn.Linear(token_feature_dim, token_embed_dim)` 投影进 Transformer。

图编码方面，只有三种 **标量 attention bias**（每层每头一个标量）：
- $s_{\phi^{SPD}(i,j)}$：无向最短路径距离
- $p_{\phi^{P}(i,j)}$：父节点有向距离
- $c_{\phi^{C}(i,j)}$：子节点有向距离

以及可选的 centrality encoding（节点度数嵌入）。

**关键：GET-Zero 没有编码的东西**

论文 Sec. V-C 明确承认：

> "It's unlikely that GET-Zero trained on LEAP hands would zero-shot transfer to a new robot hand model. For example, we train with at most 16 joints and do not encode **joint limit ranges, motor strength, friction properties or finger shape**, all of which vary with a new hand and are important in manipulation tasks."

也就是说：
1. **没有 link 几何形状**（不同手的指节长度、粗细、形状完全不同）
2. **没有关节限位**（不同手的 joint limits 差异巨大）
3. **没有 link 质量/惯性**
4. **没有电机参数**
5. 图编码只是 SPD + parent/child 标量偏置，不含边的物理属性

### 二、T(R,O) Grasp 的 embodiment 编码——一个有意义的对照

T(R,O) 虽然做的是 grasp synthesis（不是在线控制），但它的 link 几何编码方式值得借鉴：

```python
# from tro_graph.py - BPS encoding for link geometry
link_bps = self.bps.encode(link_pc, feature_type=['dists'])
link_bps = torch.cat([link_bps, centroid, scale.view(1, 1)], dim=-1)
link_embedding = self.link_token_encoder(link_bps)  # MLP: embed_dim → embed_dim
```

**BPS (Basis Point Set)**：均匀采样 B 个基点在单位球内，计算 link 点云到每个基点的最近距离，得到固定长度的几何特征。

T(R,O) 编码了完整的 link 几何但完全不编 joint-level 控制信息（因为它不做在线控制）。反过来，GET-Zero 编码了 joint-level 控制信息但几乎不编 link 几何。

### 三、GET-Zero 的实验范围——比 paper title 暗示的窄得多

从论文 Sec. IV-A 和 Fig. 4：
- 所有实验都在 **LEAP Hand family** 上
- 变体生成方式：删除关节/链接 + 1.5cm 连杆延伸
- 236 个图变体中训练了 44 个
- **所有变体共享相同的 finger layout（三主指 + 拇指）**
- **所有变体共享相同的 link 形状**（3D 打印标准件）
- 从未在 Allegro、Shadow、Barrett 或任何其他手上测试

这意味着 GET-Zero 声称的 "embodiment generalization" 实际上更像 **"topology generalization within a single hand family"**。

### 四、我们的切入空间——基于以上分析的精确定位

将 GET-Zero 的短板和 T(R,O) 的优势结合，以下是在「仅做手型泛化」前提下最值得做的事：

| 改进点 | 具体做什么 | GET-Zero 缺失的证据 | 可行性 |
|-------|----------|------------------|-------|
| **A. 跨 hand family 泛化** | 在 LEAP/Allegro/Shadow/三指手等拓扑差异大的手上验证 | 论文自认 "unlikely to transfer to a new hand model" | 高，但需要多手 URDF + 对应 RL 环境 |
| **B. 丰富的 static embodiment 编码** | 加入 link 几何（BPS 或简化版）、joint limits、rest pose 等 | 论文列出了所有缺失项 | 高，从 URDF 直接可读 |
| **C. Dynamic×Static cross-attention** | 用 cross-attn 替代简单 concat，让动态状态"查询"结构先验 | GET-Zero 把 fixed local 直接 concat 到 token 里 | 中，需要验证是否比 concat 好 |
| **D. 端到端 RL** | 跳过 per-expert BC 蒸馏，直接多手型混合 RL 训练 | GET-Zero 需要先训 44 个 expert 再蒸馏 | 中，multi-embodiment RL 需要解决 reward 对齐 |
| **E. Self-modeling loss 扩展** | 除了 FK，加上 IK 或 link-level 几何预测 | GET-Zero 只预测 FK | 低优先，边际收益不确定 |

### 五、方法叙事初步

如果要用一句话概括我们的 paper story：

> **GET-Zero 证明了 graph-biased attention 能编码运动学拓扑，但它的 embodiment 表征太弱（只有 SPD/parent-child），导致无法跨 hand family。我们通过引入 link-level 几何编码 + joint-level 物理参数作为 static embodiment stream，并用 cross-attention 让动态控制信号在结构先验下做上下文化，实现了首个真正跨 hand family 的零样本 in-hand manipulation。**

**小结**：GET-Zero 的核心价值在 graph bias + token 粒度，核心短板在 embodiment 表征太弱和泛化范围太窄。补上这两点就是一篇完整的工作。

**待确认**：4 个主要改进点（A/B/C/D）中，你认为哪些是第一版 MVP 必须包含的？

## 4.2 Graph Transformer vs MPNN：从代码出发的精确对比

**用户反馈**：不急于选改进项组合。先解决一个更底层的架构问题——Graph Transformer 还是 MPNN 更适合做主干。用户的核心判断：
1. Graph Transformer 更适合编码关节状态、输出关节空间动作
2. 但 TRO-Grasp 里的边特征比 GET-Zero 的标量 bias 更好
3. 如果 MPNN 做主干：关节为节点、边携带 SE(3) + link BPS 等，后期可扩展物体节点
4. 不希望用 zero-padding 处理变长关节输入
5. 想汲取两边的长处

**分析**：

### 一、先从代码看清两者的注意力机制到底在做什么

#### GET-Zero 的注意力（`embodiment_attention.py`）

GET-Zero 的核心计算可以从代码精确还原为：

$$
A_{ij} = \frac{Q_i K_j^T}{\sqrt{d_k}} + s_{\phi^{SPD}(i,j)} + p_{\phi^{P}(i,j)} + c_{\phi^{C}(i,j)}
$$

$$
\text{output}_i = \sum_j \text{softmax}(A_{ij}) \cdot V_j
$$

其中 $s, p, c$ 都是 `nn.Embedding` 查表得到的**每头一个标量**。也就是说：
- **图结构只影响 attention score，不影响 Value**
- **边特征极弱**：只有 3 个整数距离（SPD、父距、子距）→ 3 个标量偏置
- **Value $V_j$ 完全不含边信息**

#### TRO-Grasp 的注意力（`denoiser.py` - `GraphLayer`）

TRO-Grasp 的 RR attention 可以从代码还原为：

$$
Q_i = h_Q(V^R_i), \quad K_j = h_K(V^R_j)
$$

$$
V_{ij} = h_V(\text{concat}(V^R_i, V^R_j, E^{RR}_{ij}))
$$

$$
\alpha_{ij} = \text{softmax}\left(\frac{Q_i K_j^T}{\sqrt{d_k/H}}\right)
$$

$$
\text{out}_i = h_{\text{out}}\left(\text{concat}\left(\sum_j \alpha_{ij} V_{ij},\; \text{pool}(V_{ij})\right)\right) + V^R_i
$$

关键区别：
- **Value 是 edge-conditioned 的**：$V_{ij}$ 不只依赖 source node $j$，还依赖 edge $E^{RR}_{ij}$
- **边特征是高维向量**：$E^{RR}_{ij}$ 是 link 之间的相对 SE(3)（6 维旋转向量），经过 MLP 编码后维度与隐藏层相同
- **边特征也被更新**：每层结束后 $E^{OR}, E^{RR}$ 都会被更新
- **同时有 self-loop residual**：$\text{out}_i$ 加上 self-loop 项

### 二、两者的真正区别不是"Transformer vs MPNN"

回到代码看，TRO-Grasp 的 `GraphLayer` 其实**不是经典 MPNN**。经典 MPNN 的特征是：
1. 消息只在邻居间传递（稀疏）
2. 聚合用 sum/mean/max（无注意力权重）

而 TRO-Grasp 的做法是：
1. 全连接：$Q_i$ 对**所有** $K_j$ 计算注意力（robot-to-robot 内是全连接）
2. 聚合用 softmax attention（有权重）

所以 TRO-Grasp 更准确地说是 **Relational Attention**（关系注意力）——标准 attention 机制但 Value 里注入了边信息。

这意味着"Graph Transformer vs MPNN"这个二选一框架**不太精确**。更准确的光谱是：

| 方法 | Q/K attention | Value 中的边信息 | 边特征维度 | 边是否更新 |
|------|-------------|----------------|-----------|----------|
| GET-Zero | ✅ 全连接 + 标量 bias | ❌ | 标量 | ❌ |
| TRO-Grasp | ✅ 全连接 | ✅ edge-conditioned | 高维向量 | ✅ |
| 经典 MPNN (GATv2 等) | 邻域内 | ✅ | 高维向量 | 可选 |

### 三、你想要的东西其实是 Relational Graph Attention

综合你的 4 个诉求，你想要的架构更像：

**关节级关系注意力网络（Joint-level Relational Attention）**

1. **节点 = 关节**（继承 GET-Zero）
   - 动态特征：关节角度/速度/上步动作
   - 静态特征：joint limits、rest pose

2. **边 = 丰富的结构信息**（继承 TRO-Grasp 思想）
   - 父子关节之间的相对 SE(3)（从 URDF rest pose 精确读取）
   - 连接 link 的几何编码（BPS 或简化版本如 bbox/尺度/PCA 轴）
   - SPD / parent-child 关系（可保留 GET-Zero 的标量 bias 作为额外项）

3. **注意力机制 = Relational Attention**（结合两者）

$$
Q_i = h_Q(x^{dyn}_i, x^{stat}_i), \quad K_j = h_K(x^{dyn}_j, x^{stat}_j)
$$

$$
V_{ij} = h_V(x^{dyn}_i, x^{dyn}_j, e_{ij})
$$

$$
A_{ij} = \frac{Q_i K_j^T}{\sqrt{d_k}} + b_{\text{graph}}(i,j)
$$

$$
\text{out}_i = \sum_j \text{softmax}(A_{ij}) \cdot V_{ij}
$$

这样既保留了 GET-Zero 的**全局通信能力 + 图 bias 在 QK 上的调制**，又引入了 TRO-Grasp 的 **edge-conditioned Value**。

4. **输出 = per-joint action head**（继承 GET-Zero）

### 四、关于 zero-padding 的问题

你不想用 zero-padding。可行的替代方案：

| 方案 | 原理 | 优缺点 |
|------|------|-------|
| **PyG-style batched graph** | 把 batch 内所有手的关节拼成一个大图，用 `batch` index 区分不同手。scatter 操作天然处理变长 | ✅ 无 padding，内存高效；❌ 需要 PyG 或手写 scatter |
| **Nested Tensor**（PyTorch 2.x） | PyTorch 原生支持的变长张量 | ✅ 无 padding；❌ API 不够成熟，不是所有操作都支持 |
| **Padding + mask**（GET-Zero 做法） | `src_key_padding_mask` 屏蔽 padding token | ✅ 实现最简单；❌ 浪费计算，你不想用 |

**我的建议**：用 **PyG-style batched graph**。理由：
- PyG（`torch_geometric`）的 `Data` / `Batch` 设施非常成熟
- `scatter` / `segment_csr` 操作天然支持变长节点
- 边的定义也天然支持稀疏或全连接
- 后期扩展物体节点时，只需要在图里加新的节点类型和边类型

如果完全不想引入 PyG 依赖，也可以手写 scatter（核心就几十行）。

### 五、这个架构能否后期扩展到物体节点

可以，且很自然：

- 物体节点 = 新的节点类型
- hand↔object 边 = OR 类型边（类似 TRO-Grasp 的 OR attention）
- hand↔hand 边 = RR 类型边（当前已有）
- object↔object 边 = 不需要（刚体，内部关系固定）

也就是从 **homogeneous graph**（只有 joint nodes）扩展到 **heterogeneous graph**（joint + object nodes），TRO-Grasp 的 OR/RR 分离设计就是这个思路。

### 六、总结对比

|  | GET-Zero 风格 | TRO-Grasp 风格 | 建议的 Relational Attention |
|--|-------------|---------------|---------------------------|
| 节点 | joint tokens | link nodes | joint tokens |
| 边信息在注意力中的角色 | 标量 bias on QK | 高维向量 in V | **标量 bias on QK + 高维向量 in V** |
| 变长处理 | padding + mask | padding（固定 25） | PyG batched graph |
| link 几何 | ❌ | BPS encoding | BPS 或简化版 |
| 输出 | per-joint action | per-link SE(3) noise | per-joint action |
| 边是否更新 | ❌ | ✅ 每层更新 | 可选（第一版可不更新） |
| 可扩展到物体 | 不自然 | 天然支持（OR/RR） | 天然支持 |

**小结**：你想要的不是纯 Graph Transformer 也不是纯 MPNN，而是一种 **Joint-level Relational Attention**——节点粒度继承 GET-Zero，边的丰富度继承 TRO-Grasp，变长处理用 PyG batched graph 代替 padding。

**待确认**：这个 Relational Attention 的方向是否符合你的直觉？

## 4.3 基于用户草图的架构细化：Cross-Attention + Self-Attention 两阶段

**用户反馈**：画了架构草图，核心数据流为：
1. urdf/mjcf → MLP → 作为 cross-attention 的 **Key, Value**
2. joint state → MLP → 作为 cross-attention 的 **Query**
3. cross-attention 输出 → self-attention（hand graph 内部通信）
4. realtime kinematic graph / BPS / $E_{RR}$(pos,rot) → 作为 self-attention 的 **Bias**
5. self-attention 输出 → Policy Head → per-joint action

同时提出两个技术问题：
- GET-Zero 的 Graph Transformer 是否天然处理变长？
- Relational Attention 能否不用 zero-padding 处理变长？

**分析**：

### 一、关于变长处理的精确回答

先回答你的技术问题。你的理解基本正确但需要补充一点：

**GET-Zero 的变长处理**：它确实通过 `nn.Linear` 的性质处理了变长——因为线性层 $Y = XW + b$ 中 $W \in \mathbb{R}^{d_{in} \times d_{out}}$ 不依赖序列长度，所以任意数量的 token 都能通过同一个线性层。但是，**attention 计算本身需要知道哪些位置是有效的**。GET-Zero 的做法是 padding + `src_key_padding_mask`：

```python
# from embodiment_transformer.py
src_key_padding_mask = (torch.arange(largest_dof_count_this_batch) + 1).unsqueeze(0).repeat(batch_size, 1) > dof_counts.unsqueeze(1)
```

所以 GET-Zero **确实用了 zero-padding**，只是用 mask 让 padding 位置不参与 attention 计算。

**Relational Attention 能否无 padding？** 能。有两条路：

1. **PyG-style batched graph**：把 batch 内所有手的关节拼成一个大图。比如 batch 里有 3 只手（16, 20, 24 关节），就构造一个 60 个节点的大图，attention mask 确保 hand 1 的节点只和 hand 1 的节点交互。PyG 的 `Batch.from_data_list()` 自动处理这些。
2. **手写 scatter**：不用 PyG，手动构造 `edge_index` + `batch` index，用 `torch.scatter` 或 `torch_scatter` 实现。

但实话说，**如果你的 batch 内同一时刻只训练一种手型（最常见的多手型 RL 训练方式：每种手型一个独立 env group），那 batch 内关节数相同，padding 问题就不存在**。只有当你在同一个 batch 内混合不同 DoF 的手型时才需要处理变长。

### 二、你的草图架构分析

你的数据流可以形式化为：

**阶段 1：Cross-Attention（结构-状态对齐）**

$$
K, V = \text{MLP}_{emb}(\text{URDF}_{joints}) \in \mathbb{R}^{J \times d}
$$

$$
Q = \text{MLP}_{dyn}(\text{joint\_state}) \in \mathbb{R}^{J \times d}
$$

$$
H = \text{CrossAttn}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

这里 URDF 信息（关节上下限、rest pose、link 属性等）作为**被查询的知识库**，joint state 作为**提问者**。输出 $H$ 是「在结构先验下被上下文化的关节表征」。

**物理意义**：每个关节的当前状态 $q_j$ 在「我这个关节的活动范围是什么、我的 rest 位置在哪、我连的 link 长什么样」的上下文下被重新编码。

**阶段 2：Self-Attention + Graph Bias（关节间协调）**

$$
H' = \text{SelfAttn}(H) + \text{GraphBias}(\text{kinematic\_graph})
$$

这里 kinematic graph 特征（BPS / $E_{RR}$ 的 pos/rot）以 bias 形式注入 attention，类似 GET-Zero 的做法但 bias 可以更丰富。

**物理意义**：关节之间互相传递信息——「你手指 3 的状态对我手指 1 有什么影响？」图 bias 确保邻近关节的通信被优先保障。

**阶段 3：Policy Head**

$$
a_j = \pi(h'_j), \quad j = 1, \ldots, J
$$

### 三、这个架构的优点

1. **Cross-attention 分离了 static 和 dynamic**：URDF 是不变的先验，joint state 是实时变化的。分开编码后通过 cross-attention 融合，比 GET-Zero 直接 concat 更显式
2. **Self-attention + bias 保留了 GET-Zero 的核心贡献**：图结构通过 bias 参与通信
3. **分两阶段避免混淆**：先做「结构-状态对齐」，再做「关节间协调」

### 四、需要进一步讨论的几个细节

#### 1. Cross-Attention 的 Q/K/V 维度对齐

在你的设计中，URDF tokens 和 joint state tokens 的数量都是 $J$（关节数）。这意味着 $Q \in \mathbb{R}^{J \times d}$，$K \in \mathbb{R}^{J \times d}$，attention matrix 是 $J \times J$。

这里有个微妙问题：cross-attention 默认是「多对多」的——joint $i$ 的 query 会 attend 到**所有** $J$ 个 URDF tokens，而不只是 joint $i$ 自己的 URDF 信息。

这对不对？取决于你想要什么行为：
- **如果只让 joint $i$ 查询 joint $i$ 的 URDF**：那 cross-attention 退化为逐元素乘法/门控，不需要 attention
- **如果让 joint $i$ 也能查询其他 joint 的 URDF**：那 cross-attention 是有意义的——比如拇指关节想知道其他手指的结构约束，以便协调

我认为后者更合理，因为手型泛化需要策略「理解整只手的结构」，而不只是每个关节理解自己。

#### 2. Self-Attention 阶段的 Bias 具体长什么样

你的草图标注了 "BPS Feature / $E_{RR}$(pos, rot)" 作为 bias。这里的 bias 可以有两种注入方式：

**方式 A：GET-Zero 式标量 bias**
$$
A_{ij} = \frac{Q_i K_j^T}{\sqrt{d_k}} + f(E_{RR}(i,j))
$$

其中 $f$ 把高维边特征映射为每头一个标量。

**方式 B：向量化 bias（更丰富）**
$$
A_{ij} = \frac{(Q_i + q_{edge}(e_{ij}))(K_j + k_{edge}(e_{ij}))^T}{\sqrt{d_k}}
$$

即边特征不只影响 attention score，还直接调制 Q 和 K。

方式 B 更接近 Graphormer v2 / Relational Transformer 的做法，表达力更强。

#### 3. BPS 是放在 Static stream 还是 Bias stream

你把 BPS 放在了 self-attention 的 bias 中。但 BPS 本质上是 link 几何的静态编码——它和 URDF 一样是不变的先验。

一个替代方案是把 BPS 也放进 cross-attention 的 K/V（和 URDF 信息一起作为 static stream）。这样 cross-attention 就同时编码了：关节限位 + rest pose + link 几何形状。

self-attention 的 bias 则只保留**拓扑关系**：SPD / parent-child / 相对 SE(3)——这些更像"关系"而非"属性"。

**小结**：你的两阶段（cross-attention → self-attention + bias）数据流方向正确。关键待定的是：
1. cross-attention 到底编码哪些 static 信息
2. self-attention 的 bias 用标量还是向量
3. BPS 放在哪个 stream

**待确认**：你是否同意上面 §IV 中几个细节问题的分析？特别是 BPS 放在 cross-attention 的 static stream 还是 self-attention 的 bias 中？

## 4.4 训练范式、变长处理、Relational Attention 的精确澄清

**用户反馈**：
- BPS 放 cross-attention 的 static stream（同意）
- Cross-attention 用多对多（同意）
- 对 PyG batched graph 的说法不理解——Isaac Lab 里不同异构机器人一般不在同一 env 并行训练
- 想理解 GET-Zero 具体是怎么构建训练数据的
- 草图更像是 Graph Transformer，不是 Relational Attention——想看 Relational Attention 的 mermaid 图

**分析**：

### 一、GET-Zero 的训练范式精确还原

从论文 Sec. IV-B/C 和代码可以精确还原：

**阶段 1：训练 per-embodiment RL expert**
- 对 44 个 LEAP 变体，每个变体独立训练一个 PPO expert（在 Isaac Gym 中）
- 每个 expert 只控制自己对应的手型

**阶段 2：采集蒸馏数据集**
- Roll out 每个 expert，采集 7 小时的 demonstration data
- 数据格式：`(state, action, embodiment_id)` 三元组
- **这是一个离线数据集**

**阶段 3：BC 蒸馏**
- 从数据集中采样 mini-batch（**一个 batch 内可以混合不同手型**）
- 用 BC loss 训练统一的 GET 模型
- **正是这里才需要处理变长**——因为不同手型的 DoF 不同，一个 batch 内需要 padding + mask

所以 GET-Zero 的变长问题出现在 **BC 蒸馏阶段**，不是 RL 训练阶段。

### 二、对我们的影响：取决于训练范式选择

| 训练范式 | 是否存在变长问题 | 说明 |
|---------|------------|------|
| **GET-Zero 式：per-expert → BC 蒸馏** | ✅ 存在 | 蒸馏数据集混合了不同 DoF 的手型 |
| **端到端 RL：每种手型一个 env group** | ❌ 不存在 | 同一 env group 内 DoF 相同，不同 group 可以分别处理 |
| **端到端 RL：单手型训练，跨手型零样本** | ❌ 不存在 | 只训练一种手型，推理时换手型 |

如果我们选择端到端 RL（每种手型一个 env group），那每次 forward 的 batch 内 DoF 一致，**不需要 padding，也不需要 PyG batched graph**。线性层 $Y = XW + b$ 天然处理任意关节数。

我之前提 PyG batched graph 是在考虑 BC 蒸馏范式，如果我们走端到端 RL，这个问题不存在，这一段可以忽略。

### 三、你的草图 vs Relational Attention 的区别

你说得对，你的草图更像是 **Graph Transformer with cross-attention preprocessing**，核心特征是：
- cross-attention：static × dynamic 对齐
- self-attention + bias：关节间通信，图结构以 bias 形式进入

**Relational Attention** 的核心区别在于：**edge features 不只是 attention score 上的 bias，还直接参与 Value 的计算**。也就是说，消息内容本身也受关节间关系调制。

下面画两个 mermaid 图对比。

## 4.5 方案 A vs 方案 B：对比与选择

（见上方 mermaid 图）

### 方案 A（你的草图）：Cross-Attn → Self-Attn + Graph Bias

```
URDF+BPS → MLP → K,V  ─┐
                        ├→ Cross-Attention → Self-Attention + bias(SPD, E_RR) → Policy Head
joint state → MLP → Q ─┘
```

**特点**：
- 边特征（SPD / E_RR）只以**标量 bias** 影响 attention score
- Value 不含关节间关系
- 自注意力阶段的信息传递不受边约束调制

### 方案 B（Relational Attention）：Cross-Attn → Relational Self-Attn

```
URDF+BPS → MLP → K,V  ─┐
                        ├→ Cross-Attention → Relational Self-Attention → Policy Head
joint state → MLP → Q ─┘                           ↑
                                      E_RR(SE3) → edge MLP → V_ij = g(h_i, h_j, e_ij)
```

**特点**：
- 边特征**同时**影响 attention score（bias）和 Value（消息内容）
- $V_{ij} = g(h_i, h_j, e_{ij})$：关节 $j$ 传给关节 $i$ 的消息，受它们之间的连杆关系调制
- 这意味着：拇指传给食指的信息内容，和食指传给中指的信息内容**本质不同**（因为连接它们的 link geometry / SE(3) 不同）

### 方案 A vs B 的核心差异

| 维度 | A (Graph Transformer) | B (Relational Attention) |
|------|----------------------|--------------------------|
| 边→attention score | ✅ bias | ✅ bias |
| 边→Value内容 | ❌ | ✅ $V_{ij}=g(h_i,h_j,e_{ij})$ |
| 表达力 | 图结构只影响"看谁多/少" | 图结构同时影响"看谁多/少"和"看到什么" |
| 计算复杂度 | $O(J^2 d)$ | $O(J^2 d)$（相同，因为 $V_{ij}$ 也是 $J \times J$） |
| 实现复杂度 | 低（标准 attention + bias） | 中（需自定义 attention） |
| 物体扩展性 | 需额外设计 object interaction | 天然：加 OR attention（TRO-Grasp 模式） |

### 对第一版 MVP 的建议

两种都可行。如果你的第一版只做手型泛化：
- **方案 A 足够**：边 bias 已经能区分拓扑，GET-Zero 在 LEAP family 内已证明
- **方案 B 更好但更复杂**：消息内容受边调制，理论表达力更强，且后期扩展物体节点时不需要改主体架构

一个折中路线是：**第一版先用方案 A（更快跑通），验证 rich static stream + cross-attention 的提升；如果效果不够或准备扩展物体，再切到方案 B**。方案 A → B 的改动是局部的（只改 self-attention 层），不影响前面的 cross-attention 和后面的 policy head。

**小结**：你的草图（方案 A）已经是一个合理的第一版架构。Relational Attention（方案 B）是加强版，后期物体泛化时更有优势。两者前端（cross-attention）和后端（policy head）完全相同，差异只在 self-attention 层。

**待确认**：你倾向先用方案 A 快速验证，还是直接上方案 B？

## 4.6 图神经网络的分类谱系：术语澄清

**用户反馈**：方案 A、B 都保留。问：Relational Attention / TRO-Grasp 的 MPNN / Graphormer 这些术语之间是什么关系？

**分析**：

### 一、图神经网络（GNN）的主要家族

```
Graph Neural Networks (GNN)
├── Message Passing Neural Networks (MPNN)
│   ├── GCN (Kipf & Welling 2017)
│   ├── GAT / GATv2 (Velickovic 2018 / Brody 2022)
│   ├── GraphSAGE (Hamilton 2017)
│   └── ...
├── Graph Transformers
│   ├── Graphormer (Ying 2021)
│   ├── SAN (Spectral Attention Network)
│   ├── GraphGPS (Rampasek 2022)
│   └── ...
└── Spectral Methods (GCN via Laplacian, etc.)
```

### 二、关键术语的精确定义

**MPNN（Message Passing Neural Network）**
- Gilmer 等 2017 提出的统一框架
- 核心公式：$m_i = \text{AGG}_{j \in \mathcal{N}(i)} \phi(h_i, h_j, e_{ij})$，$h_i' = \psi(h_i, m_i)$
- 特征：**邻域聚合**，只在图的局部邻域传递消息
- GAT 是 MPNN 的一种——用 attention 做聚合权重，但仍然只在邻居间通信

**Graph Transformer**
- 核心特征：**全局 attention**——每个节点可以 attend 到所有其他节点（不限制在邻域）
- 图结构通过 bias / positional encoding / attention mask 等方式注入，而不是通过邻域限制
- Graphormer（Ying 等 2021）是最知名的一种

**Graphormer** 具体指什么
- Graphormer 是微软提出的一种 Graph Transformer
- 核心特色：$A_{ij} = \frac{Q_i K_j^T}{\sqrt{d_k}} + b_{\phi^{SPD}(i,j)}$
- 即用最短路径距离（SPD）作为 attention bias
- GET-Zero 直接借用了 Graphormer 的这个设计，并加了 parent-child bias

### 三、TRO-Grasp 的 GraphLayer 归类

TRO-Grasp 的 `GraphLayer` 比较特殊——它不是经典 MPNN（因为它做全连接 attention），也不是标准 Graph Transformer（因为 Value 是 edge-conditioned 的）。精确归类时：

| 特征 | 经典 MPNN | Graph Transformer / Graphormer | TRO-Grasp GraphLayer |
|------|----------|------------------------------|---------------------|
| 通信范围 | 邻域 | 全局 | 全局 |
| 聚合方式 | sum/mean/attention | attention | attention |
| 图信息在哪 | 约束邻域 | bias on attention score | **edge in Value** |
| Value 含边信息 | ✅（大多数 MPNN） | ❌（Graphormer） | ✅ |

TRO-Grasp 的做法更接近以下名称：
- **Relational Attention**（Shaw 等 2018）
- **Edge-Augmented Transformer**（Hussain 等 2022）
- 或更广义地称为 **Relational Graph Transformer**

这些名字都指同一类东西：**全局 attention + edge features 参与 Value/K/Q 的计算**。

### 四、我们的方案在这个谱系中的位置

| 方案 | 归类 | 最接近的文献 |
|------|------|-----------|
| GET-Zero | Graph Transformer (Graphormer-style bias) | Graphormer (Ying 2021) |
| 你的草图（方案 A） | Graph Transformer + Cross-Attention preprocessing | Graphormer + 二阶段设计 |
| Relational Attention（方案 B） | Relational Graph Transformer | Shaw 2018 / Edge-Augmented Transformer |
| TRO-Grasp | Relational Graph Attention (heterogeneous) | 异构图 + Relational Attention |

**一句话总结**：
- Graphormer = Graph Transformer，图结构以 bias 注入 attention score
- Relational Attention = attention + edge features 进入 Value/Q/K
- TRO-Grasp 的 GraphLayer 是一种异构的 Relational Attention
- GET-Zero 用的就是 Graphormer
- 你的方案 A 是 Graphormer + cross-attention
- 方案 B 是 Relational Graph Transformer + cross-attention

**小结**：如果在论文里描述方法，方案 A 可以称为"Embodiment-Conditioned Graph Transformer"（cross-attention 做 embodiment conditioning，Graphormer-style bias 做 joint communication）。方案 B 可以称为"Embodiment-Conditioned Relational Graph Transformer"。

**待确认**：你还有其他知识向的问题想在本轮讨论中澄清吗？